In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_theme(style="whitegrid")

# Definición de la carpeta de origen
folder = 'csv'

In [ ]:
dep_map = {1: 'Particular Pagado', 2: 'Part. Subvencionado', 3: 'Municipal', 4: 'SLE'}
sex_map = {1: 'Masculino', 2: 'Femenino'}

def clean_score(s):
    if pd.isna(s) or s == '': return np.nan
    if isinstance(s, str):
        s = s.replace(',', '.')
    try:
        val = float(s)
        return val if val > 0 else np.nan
    except:
        return np.nan

def load_and_merge(year, b_file, c_file):
    path_b = os.path.join(folder, b_file)
    path_c = os.path.join(folder, c_file)
    
    df_b = pd.read_csv(path_b, sep=';', low_memory=False)
    df_c = pd.read_csv(path_c, sep=';', low_memory=False)
    
    merged = pd.merge(df_b, df_c, on='ID_aux', suffixes=('', '_c'))
    merged['Year'] = year
    return merged

In [ ]:
years = [2019, 2021, 2024, 2025]
files_b = {2019: 'ArchivoB_Adm2019.csv', 2021: 'ArchivoB_Adm2021.csv', 
           2024: 'ArchivoB_Adm2024.csv', 2025: 'ArchivoB_Adm2025.csv'}
files_c = {2019: 'ArchivoC_Adm2019.csv', 2021: 'ArchivoC_Adm2021.csv', 
           2024: 'ArchivoC_Adm2024.csv', 2025: 'ArchivoC_Adm2025.csv'}

math_cols = {2019: 'MATE_ACTUAL', 2021: 'MATE_ACTUAL', 
             2024: 'MATE1_REG_ACTUAL', 2025: 'MATE1_REG_ACTUAL'}

all_data = []

for yr in years:
    df = load_and_merge(yr, files_b[yr], files_c[yr])
    m_col = math_cols[yr]
    df['Math_Score'] = df[m_col].apply(clean_score)
    df['NEM_Score'] = df['PTJE_NEM'].apply(clean_score)
    df['Dep_Name'] = df['GRUPO_DEPENDENCIA'].map(dep_map)
    df['Gender_Name'] = df['SEXO'].map(sex_map)
    
    if 'PACE' in df.columns:
        df['Is_PACE'] = df['PACE'].apply(lambda x: 1 if pd.notna(x) and x != '' and x != 0 else 0)
    else:
        df['Is_PACE'] = 0
        
    all_data.append(df[['Year', 'Math_Score', 'NEM_Score', 'Dep_Name', 'Gender_Name', 'Is_PACE', 'CODIGO_REGION']])

full_df = pd.concat(all_data, ignore_index=True)
full_df['Category_Socio'] = full_df['Dep_Name'].apply(lambda x: 'Privado (Pagado)' if x == 'Particular Pagado' else ('Público (Mun/SLE)' if x in ['Municipal', 'SLE'] else 'Subvencionado'))

In [ ]:
q1_data = full_df.groupby(['Year', 'Category_Socio'])['Math_Score'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(data=q1_data[q1_data['Category_Socio'] != 'Subvencionado'], x='Year', y='Math_Score', hue='Category_Socio', marker='o')
plt.title('Evolución de Brecha de Puntajes Matemática: Transición PSU-PAES')
plt.ylabel('Puntaje Promedio')
plt.xlabel('Año de Admisión')
plt.show()

In [ ]:
df_2025 = full_df[full_df['Year'] == 2025].dropna(subset=['NEM_Score', 'Math_Score', 'Gender_Name'])

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_2025.sample(min(2000, len(df_2025))), x='NEM_Score', y='Math_Score', hue='Gender_Name', alpha=0.4)

for gender, color in zip(['Masculino', 'Femenino'], ['blue', 'red']):
    subset = df_2025[df_2025['Gender_Name'] == gender]
    sns.regplot(data=subset.sample(min(500, len(subset))), x='NEM_Score', y='Math_Score', scatter=False, color=color, label=f'Reg. {gender}')

plt.title('Correlación NEM vs. Matemática M1 por Género (Admisión 2025)')
plt.legend()
plt.show()

In [ ]:
public_df = full_df[full_df['Category_Socio'] == 'Público (Mun/SLE)']
q3_data = public_df.groupby('Year')['Is_PACE'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=q3_data, x='Year', y='Is_PACE', color='teal')
plt.title('Proporción de Estudiantes PACE en el Sector Público')
plt.ylabel('Proporción (%)')
plt.show()

In [ ]:
# Mapeo de regiones principales para claridad visual
reg_map = {13: 'RM', 14: 'Los Ríos', 1: 'Tarapacá', 2: 'Antofagasta', 15: 'Arica', 5: 'Valparaíso', 8: 'Biobío'}
df_2025['Region'] = df_2025['CODIGO_REGION'].map(reg_map).fillna('Otras')

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_2025, x='Region', y='Math_Score', palette='Set3')
plt.title('Distribución Territorial de Puntajes Matemática (2025)')
plt.xticks(rotation=45)
plt.show()